**Telegram bot funcionando con LangSmith**
+ trazas en LangSmith
+ evaluación offline con dataset
+ evaluadores básicos de calidad


**Orden de ejecución :**
1. Ejecutar instalación
2. Ejecutar imports + funciones + configuración
3. Ejecutar evaluación LangSmith
4. Recién al final ejecutar el bot con while True

**instalar librerías**

In [2]:
!pip install --quiet openai langsmith python-dotenv jupyter_bokeh requests pandas openpyxl

In [3]:
import requests
import os
import time
import pandas as pd
from configparser import ConfigParser
from openai import OpenAI

# LangSmith
from langsmith import traceable, Client, evaluate
from langsmith.wrappers import wrap_openai


# ============================================================
# 1. CARGAR CONFIGURACIÓN
# ============================================================

parser = ConfigParser()
parser.read("/content/example.conf")

TOKEN = parser["telegram"]["TOKEN"]
API_KEY = parser["seguridad"]["OPENAI_API_KEY"]
LANGSMITH_API_KEY = parser["seguridad"]["LANGSMITH_API_KEY"]


# ============================================================
# 2. CONFIGURAR VARIABLES DE ENTORNO
# ============================================================

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "orderbot-telegram-limar"


# ============================================================
# 3. CONFIGURAR TELEGRAM Y OPENAI
# ============================================================

TELEGRAM_API_URL = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
TELEGRAM_UPDATE_URL = f"https://api.telegram.org/bot{TOKEN}/getUpdates"

client = wrap_openai(OpenAI(api_key=API_KEY))
model = "gpt-4o-mini"


# ============================================================
# 4. MENÚ
# ============================================================

MENU = {
    "helado": {
        "1/4 kilo": 2500,
        "1/2 kilo": 4000,
        "1 kilo": 7000,
    },
    "sabores": [
        "Dulce de leche", "Dulce de leche granizado", "Dulce de leche con almendras",
        "Chocolate", "Chocolate amargo", "Chocolate suizo",
        "Banana split", "Frutilla", "Frutilla a la crema",
        "Vainilla", "Limón", "Menta granizada"
    ],
    "toppings": {
        "Rocklets": 500,
        "Kit Kat": 700,
        "Oreo": 350,
    }
}


# ============================================================
# 5. PROMPT DEL SISTEMA
# ============================================================

def crear_prompt_sistema():
    return f"""
Eres OrderBot, un servicio automatizado para recolectar pedidos de Helados,
una heladería local llamada LIMAR ubicada en Av La Plata al 1500.

Primero saludas al cliente, muestras el menú y los precios.
Luego recolectas el pedido teniendo en cuenta no ofrecer ni vender nada que no esté especificado en el contexto.

Recuerda no limitar los pedidos del cliente en cuanto a cantidades y formatos.
Por ejemplo, puede pedir más de 1 kilo por gusto; aquí no hay límites.

Esperas para recolectar todo el pedido, permitiendo que agregue más helado o cualquier otra cosa del menú las veces que sea necesario.

Finalmente, cuando ya recolectaste el pedido, confirmas con el usuario la lista completa del pedido con su cuenta total,
asegurándote de hacer las cuentas correctamente.

Luego preguntas si pagará en efectivo o tarjeta.
Finalmente preguntas si es para recoger o para entregar.
Si es una entrega, pides dirección.

Finalmente, escribes literalmente:
'Los empleados le confirmarán el total (por si haya un error en mis cálculos) y le cobrarán.'

Menú:
- Cuarto kilo de helado: ${MENU['helado']['1/4 kilo']}
- Medio kilo de helado: ${MENU['helado']['1/2 kilo']}
- Un kilo de helado: ${MENU['helado']['1 kilo']}

Sabores:
{", ".join(MENU['sabores'])}

Toppings:
{chr(10).join([f"- {t}: ${MENU['toppings'][t]}" for t in MENU['toppings']])}
"""


# ============================================================
# 6. MEMORIA POR USUARIO
# ============================================================

contextos = {}

def get_user_context(chat_id):
    if chat_id not in contextos:
        contextos[chat_id] = [
            {
                "role": "system",
                "content": crear_prompt_sistema()
            }
        ]
    return contextos[chat_id]


# ============================================================
# 7. FUNCIONES DEL BOT CON TRAZAS
# ============================================================

@traceable(name="Enviar mensaje a Telegram")
def send_message(text, chat_id):
    try:
        response = requests.post(
            TELEGRAM_API_URL,
            json={
                "chat_id": chat_id,
                "text": text
            }
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al enviar mensaje: {e}")
        raise


@traceable(name="Guardar pedido en Excel")
def tickets_pedidos(chat_id, content):
    archivo_excel = "pedidos.xlsx"

    if os.path.exists(archivo_excel):
        df = pd.read_excel(archivo_excel, engine="openpyxl")
    else:
        df = pd.DataFrame(columns=["Chat_ID", "Mensaje"])

    nueva_fila = pd.DataFrame([
        {
            "Chat_ID": chat_id,
            "Mensaje": content
        }
    ])

    df = pd.concat([df, nueva_fila], ignore_index=True)
    df.to_excel(archivo_excel, index=False, engine="openpyxl")


@traceable(name="Procesar mensaje del usuario")
def process_message(user_message, chat_id):
    context = get_user_context(chat_id)

    context.append({
        "role": "user",
        "content": user_message
    })

    response = client.chat.completions.create(
        model=model,
        messages=context
    )

    bot_reply = response.choices[0].message.content

    send_message(bot_reply, chat_id)

    context.append({
        "role": "assistant",
        "content": bot_reply
    })

    if "Los empleados le confirmarán el total" in bot_reply:
        tickets_pedidos(chat_id, bot_reply)


@traceable(name="Obtener actualizaciones de Telegram")
def get_updates(offset=None):
    params = {
        "timeout": 30,
        "offset": offset
    }

    try:
        response = requests.get(
            TELEGRAM_UPDATE_URL,
            params=params
        )
        response.raise_for_status()

        data = response.json()
        return data.get("result", [])

    except requests.exceptions.RequestException as e:
        print(f"Error al obtener actualizaciones: {e}")
        raise


# ============================================================
# 8. FUNCIÓN SOLO PARA EVALUACIÓN LANGSMITH
# ============================================================
# Esta función NO envía mensajes a Telegram.
# Sirve para probar respuestas controladas.

@traceable(name="Responder para evaluación LangSmith")
def responder_para_eval(inputs: dict) -> dict:
    pregunta = inputs["pregunta"]

    mensajes = [
        {
            "role": "system",
            "content": crear_prompt_sistema()
        },
        {
            "role": "user",
            "content": pregunta
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=mensajes
    )

    respuesta = response.choices[0].message.content

    return {
        "respuesta": respuesta
    }


# ============================================================
# EVALUADORES LANGSMITH
# ============================================================

def eval_no_vende_fuera_menu(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    respuesta = outputs["respuesta"].lower()

    productos_prohibidos = [
        "pizza",
        "empanada",
        "hamburguesa",
        "café",
        "cafe",
        "gaseosa",
        "cerveza",
        "papas fritas"
    ]

    return not any(producto in respuesta for producto in productos_prohibidos)


def eval_menciona_total_si_corresponde(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    respuesta = outputs["respuesta"].lower()

    debe_mencionar_total = reference_outputs.get("debe_mencionar_total", False)

    if debe_mencionar_total:
        return "$" in respuesta or "total" in respuesta

    return True


def eval_pide_direccion_si_delivery(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    respuesta = outputs["respuesta"].lower()

    debe_pedir_direccion = reference_outputs.get("debe_pedir_direccion", False)

    if debe_pedir_direccion:
        return "dirección" in respuesta or "direccion" in respuesta

    return True


def eval_no_inventa_sabores(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    respuesta = outputs["respuesta"].lower()

    sabores_inventados = [
        "pistacho",
        "maracuyá",
        "maracuya",
        "sambayón",
        "sambayon",
        "crema americana",
        "tiramisú",
        "tiramisu"
    ]

    frases_rechazo = [
        "no tenemos",
        "no ofrecemos",
        "no vendemos",
        "no está disponible",
        "no esta disponible",
        "no se encuentra en nuestro menú",
        "no se encuentra en nuestro menu",
        "no está en nuestro menú",
        "no esta en nuestro menu"
    ]

    menciona_sabor_inventado = any(
        sabor in respuesta for sabor in sabores_inventados
    )

    rechaza_sabor = any(
        frase in respuesta for frase in frases_rechazo
    )

    if menciona_sabor_inventado and rechaza_sabor:
        return True

    if menciona_sabor_inventado and not rechaza_sabor:
        return False

    return True

# ============================================================
# 10. CREAR DATASET Y EJECUTAR EVALUACIÓN
# ============================================================

def ejecutar_evaluacion_langsmith():
    ls_client = Client()

    dataset_name = f"orderbot-limar-eval-{int(time.time())}"

    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="Evaluación offline del bot de pedidos de helado LIMAR"
    )

    examples = [
        {
            "inputs": {
                "pregunta": "Hola, quiero ver el menú."
            },
            "outputs": {
                "debe_mencionar_total": False,
                "debe_pedir_direccion": False
            }
        },
        {
            "inputs": {
                "pregunta": "Quiero 1 kilo de chocolate y dulce de leche."
            },
            "outputs": {
                "debe_mencionar_total": True,
                "debe_pedir_direccion": False
            }
        },
        {
            "inputs": {
                "pregunta": "Quiero una pizza y una gaseosa."
            },
            "outputs": {
                "debe_mencionar_total": False,
                "debe_pedir_direccion": False
            }
        },
        {
            "inputs": {
                "pregunta": "Quiero medio kilo de frutilla a domicilio."
            },
            "outputs": {
                "debe_mencionar_total": True,
                "debe_pedir_direccion": True
            }
        },
        {
            "inputs": {
                "pregunta": "Quiero un kilo de limón con Oreo y Kit Kat."
            },
            "outputs": {
                "debe_mencionar_total": True,
                "debe_pedir_direccion": False
            }
        },
        {
            "inputs": {
                "pregunta": "Tenés helado de pistacho?"
            },
            "outputs": {
                "debe_mencionar_total": False,
                "debe_pedir_direccion": False
            }
        }
    ]

    for example in examples:
        ls_client.create_example(
            dataset_id=dataset.id,
            inputs=example["inputs"],
            outputs=example["outputs"]
        )

    resultados = evaluate(
        responder_para_eval,
        data=dataset_name,
        evaluators=[
            eval_no_vende_fuera_menu,
            eval_menciona_total_si_corresponde,
            eval_pide_direccion_si_delivery,
            eval_no_inventa_sabores
        ],
        experiment_prefix="eval-orderbot-limar"
    )

    return resultados

In [4]:
resultados = ejecutar_evaluacion_langsmith()

View the evaluation results for experiment: 'eval-orderbot-limar-ca6d4f65' at:
https://smith.langchain.com/o/9f5470b8-1514-4770-9640-1f13aa29cdc9/datasets/d6a23d63-ac48-4adb-97c3-d552df078866/compare?selectedSessions=cafe5528-35c2-4cab-868e-b5214f74d020




0it [00:00, ?it/s]

In [5]:
last_update_id = None

while True:
    updates = get_updates(last_update_id)

    for update in updates:
        if "message" in update:
            chat_id = update["message"]["chat"]["id"]
            user_message = update["message"].get("text", "")

            if user_message:
                process_message(user_message, chat_id)

            last_update_id = update["update_id"] + 1

    time.sleep(2)

KeyboardInterrupt: 

In [6]:
from langsmith import Client

ls_client = Client()

datasets = list(ls_client.list_datasets())

for d in datasets:
    if "orderbot-limar-eval" in d.name:
        print(d.name)

orderbot-limar-eval-1782144544
orderbot-limar-eval-1782131929
orderbot-limar-eval-1781912945


In [7]:
from langsmith import Client

ls_client = Client()

projects = list(ls_client.list_projects())

for p in projects:
    if "eval-orderbot-limar" in p.name or "orderbot" in p.name.lower():
        print(p.name)

eval-orderbot-limar-ca6d4f65
eval-orderbot-limar-663eb1ed
eval-orderbot-limar-d1699540
orderbot-telegram-limar


In [8]:
from langsmith import Client

ls_client = Client()

runs = list(ls_client.list_runs(
    project_name="eval-orderbot-limar-d1699540",
    limit=20
))

for r in runs:
    print("RUN:", r.name)
    print("INPUTS:", r.inputs)
    print("OUTPUTS:", r.outputs)
    print("ERROR:", r.error)
    print("-" * 50)

RUN: ChatOpenAI
INPUTS: {'messages': [{'content': "\nEres OrderBot, un servicio automatizado para recolectar pedidos de Helados,\nuna heladería local llamada LIMAR ubicada en Av La Plata al 1500.\n\nPrimero saludas al cliente, muestras el menú y los precios.\nLuego recolectas el pedido teniendo en cuenta no ofrecer ni vender nada que no esté especificado en el contexto.\n\nRecuerda no limitar los pedidos del cliente en cuanto a cantidades y formatos.\nPor ejemplo, puede pedir más de 1 kilo por gusto; aquí no hay límites.\n\nEsperas para recolectar todo el pedido, permitiendo que agregue más helado o cualquier otra cosa del menú las veces que sea necesario.\n\nFinalmente, cuando ya recolectaste el pedido, confirmas con el usuario la lista completa del pedido con su cuenta total,\nasegurándote de hacer las cuentas correctamente.\n\nLuego preguntas si pagará en efectivo o tarjeta.\nFinalmente preguntas si es para recoger o para entregar.\nSi es una entrega, pides dirección.\n\nFinalmente,

In [9]:
#ver el dataset
from langsmith import Client

ls_client = Client()

examples = list(ls_client.list_examples(
    dataset_name="orderbot-limar-eval-1781912945"
))

for ex in examples:
    print("INPUT:", ex.inputs)
    print("EXPECTED:", ex.outputs)
    print("-" * 50)

INPUT: {'pregunta': 'Tenés helado de pistacho?'}
EXPECTED: {'debe_mencionar_total': False, 'debe_pedir_direccion': False}
--------------------------------------------------
INPUT: {'pregunta': 'Quiero un kilo de limón con Oreo y Kit Kat.'}
EXPECTED: {'debe_mencionar_total': True, 'debe_pedir_direccion': False}
--------------------------------------------------
INPUT: {'pregunta': 'Quiero medio kilo de frutilla a domicilio.'}
EXPECTED: {'debe_mencionar_total': True, 'debe_pedir_direccion': True}
--------------------------------------------------
INPUT: {'pregunta': 'Quiero una pizza y una gaseosa.'}
EXPECTED: {'debe_mencionar_total': False, 'debe_pedir_direccion': False}
--------------------------------------------------
INPUT: {'pregunta': 'Quiero 1 kilo de chocolate y dulce de leche.'}
EXPECTED: {'debe_mencionar_total': True, 'debe_pedir_direccion': False}
--------------------------------------------------
INPUT: {'pregunta': 'Hola, quiero ver el menú.'}
EXPECTED: {'debe_mencionar_tot